<a href="https://colab.research.google.com/github/BarGinger/HCML-NLP-Project/blob/main/src/proto-lm/drugs_reviews_proto_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Using Proto-ML [1] to analyse the drug reviews dataset [2]

[1] https://github.com/yx131/proto-lm/tree/main

[2] https://www.kaggle.com/datasets/mohamedabdelwahabali/drugreview/data

### Global imports

In [1]:
import argparse
!pip install pytorch-lightning
!pip install -U datasets
import torch
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification
from ProtoLM import proto_lm
from proto_data_class import sst_datamodule
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers.tensorboard import TensorBoardLogger
import datasets

In [2]:
from transformers import AutoConfig

# Parameters for Proto-LM training on the drug review dataset
model_name = 'bert-base-uncased'  # Backbone LLM model to load
# Maximum sentence length to pad/truncate to
args = {
    'model_name': model_name,        # backbone LLM model to load
    'max_seq_length': 100,                # maximum sentence length to pad/truncate to
    'num_prototypes': 1000,               # number of prototypes to train
    'hidden_shape': 1024,                # hidden shape of each prototype, should match LLM output
    'num_classes': 10,                    # Changed to 1 for regression (predicting continuous rating 1-10)
    'cohsep_ratio': 0.5,                 # ratio of prototypes in class to push/pull
    'lambda0': 0.5,                      # lambda0 in loss
    'lr': 3e-4,                          # initial learning rate
    'proto_training_weights': 1,         # whether to train prototype weights (1=True, 0=False)
    'batch_size': 128,                   # batch size for dataloader
    'logger_dir': 'tb_logs',             # directory for the logger to store training details
    'checkpoint_dir': 'ckpt_dir',        # directory to store checkpoints
    'config_subdir': 'config_subdir',    # subdirectory for checkpoints of a certain config
    'max_epochs': 15,                     # number of epochs to train
    'num_gpu': 4,                        # number of gpus to train on
    'load_model': model_name,  # path to load a pretrained model, if any
}



# Dynamically fetch hidden size from the model configuration
config = AutoConfig.from_pretrained(args['model_name'])
hidden_size = config.hidden_size  # Dynamically get the hidden size (768 for bert-base-uncased)
args['hidden_shape'] = hidden_size

print(f'args: {args}')


# get data module
drug_review_dm = sst_datamodule(
    model_name_or_path=args['model_name'],
    max_seq_length=args['max_seq_length'],
    train_batch_size=args['batch_size'],
    eval_batch_size=args['batch_size']
)
drug_review_dm.setup(stage='fit')

print(f"loading a model: {args['load_model']}")

base_model = AutoModelForSequenceClassification.from_pretrained(
    args['model_name'], ignore_mismatched_sizes=True
)

if hasattr(base_model, "roberta"):
    llm_model = base_model.roberta
elif hasattr(base_model, "bert"):
    llm_model = base_model.bert
else:
    llm_model = base_model

proto = proto_lm(
    pretrained_model=llm_model,
    max_seq_length=args['max_seq_length'],
    num_prototypes=args['num_prototypes'],
    hidden_shape=args['hidden_shape'],
    num_classes=args['num_classes'],
    cohsep_ratio=args['cohsep_ratio'],
    lambda0=args['lambda0'],
    lr=args['lr'],
    proto_training_weights=bool(args['proto_training_weights']),
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

args: {'model_name': 'bert-base-uncased', 'max_seq_length': 100, 'num_prototypes': 1000, 'hidden_shape': 768, 'num_classes': 10, 'cohsep_ratio': 0.5, 'lambda0': 0.5, 'lr': 0.0003, 'proto_training_weights': 1, 'batch_size': 128, 'logger_dir': 'tb_logs', 'checkpoint_dir': 'ckpt_dir', 'config_subdir': 'config_subdir', 'max_epochs': 15, 'num_gpu': 4, 'load_model': 'bert-base-uncased'}


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

self.columns: ['sentiment_neg', 'sentiment_neu', 'sentiment_pos', 'sentiment_compound', 'input_ids', 'token_type_ids', 'attention_mask', 'labels', 'sentiment_features']
split is: test


Map:   0%|          | 0/46108 [00:00<?, ? examples/s]

self.columns: ['sentiment_neg', 'sentiment_neu', 'sentiment_pos', 'sentiment_compound', 'input_ids', 'token_type_ids', 'attention_mask', 'labels', 'sentiment_features']
loading a model: bert-base-uncased


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# get training utilities like logger and checkpoints
from pytorch_lightning.loggers.tensorboard import TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint

tb_logger = TensorBoardLogger(f"{args['logger_dir']}", name="drug_review_tensorboard_logs")
ckpt_path = f"{args['checkpoint_dir']}/{args['config_subdir']}"
checkpoint_callback = ModelCheckpoint(
    dirpath=ckpt_path,
    monitor='val_loss',
    save_top_k=3,
    filename="{epoch}-{val_loss:.4f}-{val_accuracy:.4f}"
)

# get trainer object
trainer = pl.Trainer(
    max_epochs=args['max_epochs'],
    accelerator="auto",
    devices=1,  # or just remove this line for auto
    logger=tb_logger,
    callbacks=[checkpoint_callback]
)

trainer.fit(proto, datamodule=drug_review_dm)

# Optionally test or save misclassified
# trainer.test(proto, datamodule=drug_review_dm, ckpt_path=args['load_model'])
# torch.save(proto.misclassified, 'drug_review_logs/misclassed.pt')

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.utilities.rank_zero:You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


split is: train


Map:   0%|          | 0/110811 [00:00<?, ? examples/s]

/content/proto_data_class.py:168: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sentiment_neg = torch.tensor(example_batch["sentiment_neg"], dtype=torch.float32)
/content/proto_data_class.py:169: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sentiment_neu = torch.tensor(example_batch["sentiment_neu"], dtype=torch.float32)
/content/proto_data_class.py:170: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sentiment_pos = torch.tensor(example_batch["sentiment_pos"], dtype=torch.float32)
/content/proto_data_class.py:171: UserWarning: To copy con

self.columns: ['sentiment_neg', 'sentiment_neu', 'sentiment_pos', 'sentiment_compound', 'input_ids', 'token_type_ids', 'attention_mask', 'labels', 'sentiment_features']
split is: validation


...

/usr/local/lib/python3.11/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

### Save the trained model

In [ ]:
# Save the complete model for deployment and reproducibility
import os
import json
from datetime import datetime

# Create a timestamped save directory
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_dir = f"final_proto_model_{timestamp}"
os.makedirs(save_dir, exist_ok=True)

print(f"Saving complete model to: {save_dir}")

# 1. Save the PyTorch Lightning checkpoint (includes everything)
trainer.save_checkpoint(os.path.join(save_dir, "model_checkpoint.ckpt"))

# 2. Save just the model weights (for loading into other frameworks)
torch.save(proto.state_dict(), os.path.join(save_dir, "model_weights.pt"))

# 3. Save the tokenizer (essential for preprocessing)
tokenizer = drug_review_dm.tokenizer
tokenizer.save_pretrained(os.path.join(save_dir, "tokenizer"))

# 4. Save model configuration and training arguments
model_config = {
    "model_name": args['model_name'],
    "max_seq_length": args['max_seq_length'],
    "num_prototypes": args['num_prototypes'],
    "hidden_shape": args['hidden_shape'],
    "num_classes": args['num_classes'],
    "cohsep_ratio": args['cohsep_ratio'],
    "lambda0": args['lambda0'],
    "lr": args['lr'],
    "proto_training_weights": args['proto_training_weights'],
    "batch_size": args['batch_size'],
    "max_epochs": args['max_epochs'],
    "timestamp": timestamp,
    "model_type": "ProtoLM_regression"
}

with open(os.path.join(save_dir, "model_config.json"), "w") as f:
    json.dump(model_config, f, indent=2)

# 5. Save data preprocessing info
data_config = {
    "text_fields": drug_review_dm.text_fields,
    "num_labels": drug_review_dm.num_labels,
    "max_seq_length": drug_review_dm.max_seq_length,
    "loader_columns": drug_review_dm.loader_columns,
    "model_name_or_path": drug_review_dm.model_name_or_path
}

with open(os.path.join(save_dir, "data_config.json"), "w") as f:
    json.dump(data_config, f, indent=2)

# 6. Save training metrics/logs if available
try:
    if hasattr(trainer.logger, 'log_dir'):
        import shutil
        shutil.copytree(trainer.logger.log_dir, os.path.join(save_dir, "logs"))
except:
    print("Could not copy training logs")

print(f"✅ Model saved successfully!")
print(f"📁 Save directory: {save_dir}")
print(f"📋 Contents:")
print(f"   - model_checkpoint.ckpt (PyTorch Lightning checkpoint)")
print(f"   - model_weights.pt (PyTorch state dict)")
print(f"   - tokenizer/ (Hugging Face tokenizer)")
print(f"   - model_config.json (model configuration)")
print(f"   - data_config.json (data preprocessing config)")
print(f"   - logs/ (training logs, if available)")

In [ ]:
!zip -r /content/final_proto_model_20250630_122824.zip /content/final_proto_model_20250630_122824


### Calc Quantus Metrics

In [ ]:
# Evaluate regression performance
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch

model = proto
model.eval()
all_predictions = []
all_labels = []

# Get predictions on test set
with torch.no_grad():
    for batch in drug_review_dm.test_dataloader():
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        sentiment_features = batch["sentiment_features"]
        labels = batch["labels"]

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            sentiment_features=sentiment_features
        )

        predictions = outputs['logits'].squeeze(-1)  # Remove last dimension for regression

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)

# Calculate regression metrics
mse = mean_squared_error(all_labels, all_predictions)
mae = mean_absolute_error(all_labels, all_predictions)
r2 = r2_score(all_labels, all_predictions)
rmse = np.sqrt(mse)

print("Regression Performance Metrics:")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R² Score: {r2:.4f}")

print(f"\nPrediction Statistics:")
print(f"Predicted range: {all_predictions.min():.2f} to {all_predictions.max():.2f}")
print(f"Actual range: {all_labels.min():.2f} to {all_labels.max():.2f}")
print(f"Mean predicted: {all_predictions.mean():.2f}")
print(f"Mean actual: {all_labels.mean():.2f}")

# Plot predictions vs actual
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

# Scatter plot
plt.subplot(1, 2, 1)
plt.scatter(all_labels, all_predictions, alpha=0.6)
plt.plot([1, 10], [1, 10], 'r--', label='Perfect Prediction')
plt.xlabel('Actual Rating')
plt.ylabel('Predicted Rating')
plt.title('Predicted vs Actual Ratings')
plt.legend()
plt.grid(True)

# Residuals plot
plt.subplot(1, 2, 2)
residuals = all_predictions - all_labels
plt.scatter(all_labels, residuals, alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Actual Rating')
plt.ylabel('Residuals (Predicted - Actual)')
plt.title('Residuals Plot')
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import quantus
import torch
import numpy as np
import pandas as pd

# Define your model and data
model = proto  # Your Proto-LM model
model_name_for_csv = f"ProtoLM_{model_name}"  # Add your model name here
model.eval()  # Set the model to evaluation mode

# Define a wrapper for your model to work with Quantus
class ModelWrapper:
    def __init__(self, model):
        self.model = model

    def __call__(self, input_ids, attention_mask):
        with torch.no_grad():
            outputs = self.model.LLM(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True
            )
            logits = outputs.logits  # Assuming logits are the output
        return logits

wrapped_model = ModelWrapper(model)

# Define a sample batch of data (input_ids and attention_mask)
batch = next(iter(drug_review_dm.test_dataloader()))
input_ids = batch["input_ids"]
attention_mask = batch["attention_mask"]
sentiment_features = torch.cat((
    batch["sentiment_neg"],
    batch["sentiment_neu"],
    batch["sentiment_pos"],
    batch["sentiment_compound"]
), dim=1)
labels = batch["labels"]

# Pass sentiment features to the model
outputs = proto(
    input_ids=input_ids,
    attention_mask=attention_mask,
    sentiment_features=sentiment_features,
    labels=labels
)

# Define an attribution method (e.g., Integrated Gradients)
from captum.attr import IntegratedGradients
ig = IntegratedGradients(wrapped_model)

# Generate attributions for the input
attributions = ig.attribute(inputs=input_ids, additional_forward_args=(attention_mask,), target=labels)

# Convert attributions to numpy for Quantus
attributions_np = attributions.detach().cpu().numpy()

# Define Quantus metrics
metrics = {
    "Sparsity": quantus.Sparsity(),
    "Complexity": quantus.Complexity(),
    "Faithfulness": quantus.FaithfulnessCorrelation(),
    "Robustness": quantus.LocalLipschitzEstimate(),
    "Sensitivity": quantus.SensitivityN()
}

# Evaluate metrics
results = {}
for metric_name, metric in metrics.items():
    result = metric(
        model=wrapped_model,
        x_batch=input_ids.cpu().numpy(),
        y_batch=labels.cpu().numpy(),
        a_batch=attributions_np,
        explain_func=lambda x: attributions_np  # Use precomputed attributions
    )
    results[metric_name] = result

# Print results
for metric_name, result in results.items():
    print(f"{metric_name}: {result}")

# Export results to CSV
results_df = pd.DataFrame.from_dict(results, orient="index", columns=["Score"])
results_df["Model"] = model_name_for_csv  # Add model name to the DataFrame
results_df.reset_index(inplace=True)
results_df.rename(columns={"index": "Metric"}, inplace=True)

# Save to CSV
csv_filename = f"quantus_metrics_{model_name_for_csv}.csv"
results_df.to_csv(csv_filename, index=False)
print(f"Results saved to {csv_filename}")

### Plot similarity between test set cases and the learned concepts similar to figure 3 in their paper

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F

# Get prototype vectors (concepts)
prototypes = proto.prototypes.detach().cpu().numpy()  # shape: (num_prototypes, hidden_dim)

# Get representations for some samples (e.g., from your test set)
batch = next(iter(drug_review_dm.test_dataloader()))
with torch.no_grad():
    # Get last hidden states for the batch
    llm_out = proto.LLM(
        input_ids=batch['input_ids'].to(proto.device),
        attention_mask=batch['attention_mask'].to(proto.device),
        output_hidden_states=True
    )
    sample_reps = llm_out.hidden_states[-1][:, 0, :].cpu().numpy()  # [CLS] token or mean pooling



# prototypes: (num_prototypes, hidden_dim)
# sample_reps: (batch_size, hidden_dim)
# Assume proto.prototype_class_vec exists and is (num_prototypes, num_classes)

# 1. Identify positive and negative prototypes
# If proto.prototype_class_vec is one-hot or softmax over classes:
proto_class = proto.prototype_class_vec.detach().cpu().numpy()  # (num_prototypes, num_classes)
positive_proto_idx = np.argmax(proto_class[:, 1])  # class 1 = positive
negative_proto_idx = np.argmax(proto_class[:, 0])  # class 0 = negative

positive_proto = torch.tensor(prototypes[positive_proto_idx])
negative_proto = torch.tensor(prototypes[negative_proto_idx])

# 2. Compute similarities for each sample
sample_vecs = torch.tensor(sample_reps)  # (batch_size, hidden_dim)
sim_pos = F.cosine_similarity(sample_vecs, positive_proto.unsqueeze(0), dim=1)
sim_neg = F.cosine_similarity(sample_vecs, negative_proto.unsqueeze(0), dim=1)

# 3. Get ground-truth labels for the batch
labels = batch['labels'].cpu().numpy()

# 4. Plot
plt.figure(figsize=(8, 8))
for label in np.unique(labels):
    idxs = np.where(labels == label)[0]
    plt.scatter(sim_pos[idxs], sim_neg[idxs], label=f"Class {label}", alpha=0.7)
plt.xlabel("Similarity to Positive Prototype")
plt.ylabel("Similarity to Negative Prototype")
plt.title("2D Prototypical Space (like Proto-LM Fig. 3)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import torch.nn.functional as F

sample_vec = sample_reps[0]  # pick one sample
proto_vecs = torch.tensor(prototypes)
similarities = F.cosine_similarity(torch.tensor(sample_vec).unsqueeze(0), proto_vecs)
plt.bar(range(len(similarities)), similarities.numpy())
plt.xlabel("Prototype Index")
plt.ylabel("Cosine Similarity")
plt.title("Sample-Prototype Similarity")
plt.show()

In [ ]:
import torch
import pandas as pd
import numpy as np
import torch.nn.functional as F

# 1. Get all review texts and their embeddings from the training set
train_dataset = drug_review_dm.dataset["train"]
all_texts = train_dataset["review"]

# Get all input_ids and attention_mask for the train set
input_ids = train_dataset["input_ids"]
attention_mask = train_dataset["attention_mask"]

# Compute all embeddings (CLS token)
all_embeddings = []
batch_size = 128
for i in range(0, len(input_ids), batch_size):
    batch_input_ids = input_ids[i:i+batch_size].to(proto.device)
    batch_attention_mask = attention_mask[i:i+batch_size].to(proto.device)
    with torch.no_grad():
        outputs = proto.LLM(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_mask,
            output_hidden_states=True
        )
        batch_embeds = outputs.hidden_states[-1][:, 0, :].cpu()  # CLS token
        all_embeddings.append(batch_embeds)
all_embeddings = torch.cat(all_embeddings, dim=0)  # (num_samples, hidden_dim)

# 2. For each prototype, find the closest text
prototypes = proto.prototypes.detach().cpu()  # (num_prototypes, hidden_dim)
closest_texts = []
for proto_vec in prototypes:
    sims = F.cosine_similarity(all_embeddings, proto_vec.unsqueeze(0), dim=1)
    idx = torch.argmax(sims).item()
    closest_texts.append(all_texts[idx])

# 3. Save to CSV
df = pd.DataFrame({"prototype_index": range(len(closest_texts)), "closest_text": closest_texts})
df.to_csv("prototype_texts.csv", index=False)